### Oct 31, 2025

On local machine:
- While in `projworks` directory, run `./package4chtc.sh sophia-lakeview` to generate `CHTC/sophia-lakeview.zip`.
- While in `projworks` directory, transfer to staging on CHTC: `scp CHTC/sophia-lakeview.zip pravindran@transfer.chtc.wisc.edu:/staging/pravindran/tonode`.


On staging:
- `ssh pravindran@transfer.chtc.wisc.edu`
- Create `/staging/pravindran/fromnode/sophia-lakeview/<model, deploy>`: model/deploy outputs transferred here from compute node.

On Townsend submit node:
- `cd sophia-lakeview`.
- `python cleanup.py` - this creates empty `err`, `log` and `out` directories.
- Change contents of `submit_trait_train_internal_deploy.sub` for number of jobs.
- Change contents of `run_trait_train_internal_deploy.sh` for config parameters.
- `condor_submit submit_trait_train_internal_deploy.sub`

---

#### `package4chtc.sh`:
```
#!/bin/bash 

# cd into projworks before running this!

PROJECTNAME=$1

zip -r hytraits.zip hytraits;
zip -r $PROJECTNAME.zip $PROJECTNAME;

if [ -d CHTC/$PROJECTNAME ]; then
  rm -rf CHTC/$PROJECTNAME  
fi
  
mkdir CHTC/$PROJECTNAME;
mv hytraits.zip CHTC/$PROJECTNAME;
mv $PROJECTNAME.zip CHTC/$PROJECTNAME;
cp CHTC/common/hytraitsenv.tar.gz CHTC/$PROJECTNAME;

cd CHTC;
zip -r $PROJECTNAME.zip $PROJECTNAME;
rm -rf $PROJECTNAME;
cd ..;
```

---

#### `submit_trait_train_internal_deploy.sub`:
```
# submit_trait_train_internal_deploy.sub
universe = vanilla

# IMPORTANT! Require execute servers that have Staging:
Requirements = (Target.HasCHTCStaging == true)

# Set files to capture log, standard output & error
log = $(Cluster)_$(Process).log
error = $(Cluster)_$(Process).err
output = $(Cluster)_$(Process).out

# Specify executable
executable = ./run_trait_train_internal_deploy.sub.sh

# Arguments to pass to the executable
arguments = $(Process)

should_transfer_files = YES
getenv = TRUE

request_cpus = 1
request_disk = 16GB
request_memory = 8GB

queue 1 # change to required number of jobs
```

---

#### `run_trait_train_internal_deploy.sh`:

```
!/bin/bash

USERNAME=pravindran
PROJECTNAME=sophia-lakeview
ENVNAME=hytraitsenv
CONFIGIDX="$1"

# transfer project from staging
echo 'Transferring: staging --> remote ...'
cp /staging/$USERNAME/tonode/$PROJECTNAME.zip $HOME

# extract project in $HOME
echo "Extractng project package ..."
unzip -qq $HOME/$PROJECTNAME.zip -d $HOME
rm -f $HOME/$PROJECTNAME.zip 
mv $HOME/$PROJECTNAME/$ENVNAME.tar.gz $HOME
mv $HOME/$PROJECTNAME/$PROJECTNAME.zip $HOME 
mv $HOME/$PROJECTNAME/hytraits.zip $HOME
rm -rf $HOME/$PROJECTNAME

# hytraits
echo "--- Extracting hytraits.zip ..."
unzip -qq $HOME/hytraits.zip -d $HOME
rm -f $HOME/hytraits.zip

echo "--- Extracting ${PROJECTNAME} ..."
unzip -qq $HOME/$PROJECTNAME.zip -d $HOME
rm -f $HOME/$PROJECTNAME.zip

# conda environment
echo "--- Extracing hytraitsenv ..."
mkdir $HOME/$ENVNAME
tar -xzf $HOME/$ENVNAME.tar.gz -C $HOME/$ENVNAME
rm $HOME/$ENVNAME.tar.gz
export PATH=$HOME/$ENVNAME:$HOME/$ENVNAME/lib:$HOME/$ENVNAME/share:$PATH
source $HOME/$ENVNAME/bin/activate
conda-unpack

# process
echo "Processing ..."
# change next line appropriately.
python $HOME/$PROJECTNAME/setup.py --n_inners 10 --n_outers 20 --internal_deploy
python $HOME/$PROJECTNAME/train.py --config_idx "${CONFIGIDX}"
python $HOME/$PROJECTNAME/deploy.py --config_idx "${CONFIGIDX}"

# zip files for transfer to staging
MODELSDIR=$HOME/$PROJECTNAME/io/model
for modeldir in "$MODELSDIR"/*/ ; do
    if [ -d "$modeldir" ]; then
        name=$(basename $modeldir)
        echo "Zipping model: ${name} ..."
        zip -qr $MODELSDIR/$name.zip $modeldir
    fi
done


DEPLOYSDIR=$HOME/$PROJECTNAME/io/deploy
for deploydir in "$DEPLOYSDIR"/*/ ; do
    if [ -d "$deploydir" ]; then
        name=$(basename $deploydir)
        echo "Zipping deploy: ${name} ..."
        zip -qr $DEPLOYSDIR/$name.zip $deploydir
    fi
done

# transfer zip file to staging
echo 'Transferring remote --> staging ...'
cp $MODELSDIR/*.zip /staging/$USERNAME/fromnode/$PROJECTNAME/model
cp $DEPLOYSDIR/*.zip /staging/$USERNAME/fromnode/$PROJECTNAME/deploy

exit
exit
```

#### Version notes

- `setup_o1.*`: Works with the first version of DataFrame that Sophia provided. The CSV with the `_v2` suffix is the version used when making entry on Oct 31, 2025 above.